# How to create a candidate panel

A **candidate panel** is a flat table of tradeable pair-spread candidates: one row per (as-of date × spread × weight model), each carrying its hedge weights and mean-reversion diagnostics (`kappa`, `half_life`, `mr_score`, `adf_pvalue`, `is_valid`, ...). It is produced by walking forward over a sector's return history, fitting a causal residual model at each date, and scoring every pair. This notebook builds one panel for the **Materials** sector using the repo's canonical residual configuration, persists it to disk, and loads it back.

The entry point is `run_panel_batch(PanelBatchConfig(...))` from `src.candidates.panel_batch` — the same function the `residuals` stage of `run_me.py` calls.

## Canonical reference configuration

The `PanelBatchConfig` below reproduces the residual settings used by `run_me.py::_make_panel_batch_cfg`. These four fields together define the residual key **`exp_hl504_mh1008_rf`** (the `CausalResidualConfig.key` that downstream simulation matches against):

| field | value | contributes |
| --- | --- | --- |
| `residual_window_mode` (default) | `"expanding"` | `exp` |
| `TimescaleConfig(residual_half_life=504)` | `504` | `hl504` |
| `residual_min_history_multiplier=2` | `504 * 2 = 1008` | `mh1008` |
| `subtract_risk_free=True` | | `rf` |

> **Note on `zlb=126`:** the z-score lookback (`z_score.lookback` in `demo_materials.yaml`) is a *simulation* parameter consumed later by `ZScoreConfig`. It plays no part in panel creation and is intentionally absent here.

`max_steps` is set purely to keep this notebook fast — it caps the number of W-FRI panel dates. The production run in `run_me.py` omits it and covers the full history.

In [1]:
from pathlib import Path

from src.candidates.panel_batch import PanelBatchConfig, run_panel_batch
from src.simulator.config import TimescaleConfig

cfg = PanelBatchConfig(
    timescales=[TimescaleConfig(residual_half_life=504)],
    subtract_risk_free=True,
    residual_min_history_multiplier=2,   # min_history = 504 * 2 = 1008
    selected_sectors=["materials"],
    frequency="W-FRI",
    max_steps=8,                         # notebook-only cap; omit for full history
    persist_result=True,
    persist_residual_params=True,
    persist_dir_template="howto",        # subdir under settings.CANDIDATE_PANELS_ROOT
)

# resolved_universe_dir()/resolved_data_path() fall back to settings when unset.
results = run_panel_batch(cfg)

Batch 20260720_1120
Sectors: 1 [mat]
Timescales: ['rhl504_zhl84']
Total jobs: 1


Sectors:   0%|          | 0/1 [00:00<?, ?sector/s]

Loading universe materials_only_v1...


  mat / rhl504_zhl84: 624/624 valid candidates

Done. 1 panels created.


`run_panel_batch` returns a dict keyed by `(group_id, timescale_label)`. Each value is a `CandidatePanelResult` with a `.panel` DataFrame and a `.metadata` dict.

In [2]:
(group_id, ts_label), result = next(iter(results.items()))
panel = result.panel

print(f"key: {(group_id, ts_label)}")
print(f"rows: {len(panel)}  valid: {int(panel['is_valid'].sum())}")
print(f"residual key: {result.metadata['residual_cfg']}")
panel[["asof_date", "spread_id", "weight_model", "kappa", "half_life", "mr_score", "is_valid"]].head()

key: ('materials', 'rhl504_zhl84')
rows: 624  valid: 624
residual key: {'window_mode': 'expanding', 'lookback': 252, 'half_life': 504, 'min_history': 1008, 'remove_residual_pcs': 0, 'subtract_risk_free': True}


,asof_date,spread_id,weight_model,kappa,half_life,mr_score,is_valid
0,2009-08-14,APD|AVY,pca,0.027276,25.411931,1.455425,True
1,2009-08-14,APD|BALL,pca,0.011562,59.948202,0.085537,True
2,2009-08-14,APD|CF,pca,0.041635,16.648337,2.312385,True
3,2009-08-14,APD|ECL,pca,0.088376,7.843182,0.564488,True
4,2009-08-14,APD|IFF,pca,0.055329,12.527631,2.335524,True


## Loading a persisted panel back

Persistence writes three artifacts per panel stem into the subdir: `{stem}.panel.parquet`, `{stem}.meta.json`, and `{stem}_residual_params.pkl`. `run_panel_batch` generates the timestamped stem internally, so recover it from the result metadata (the same trick `run_me.py::_extract_panel_stem` uses), then reload with `load_candidate_panel_result`.

In [3]:
from src.candidates.candidate_panel import load_candidate_panel_result

out_dir = Path(result.metadata["artifact_out_dir"])
stem = Path(result.metadata["residual_params_path"]).name.removesuffix("_residual_params.pkl")

reloaded = load_candidate_panel_result(out_dir=out_dir, stem=stem)
print(f"out_dir: {out_dir}")
print(f"stem:    {stem}")
print(f"reloaded panel: {reloaded.panel.shape}, valid: {int(reloaded.panel['is_valid'].sum())}")

out_dir: /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto
stem:    mat_pairs_pca_W-FRI_exp_hl504_20260720_1120
reloaded panel: (624, 18), valid: 624


## Out of scope: stock-residual & spread-level series

This notebook stops at the candidate panel (plus its fitted daily residual params). Precomputing and persisting the **stock-residual and spread-level series** — which the simulator later loads instead of recomputing residuals each step — is a **separate step** and is not covered here.

That step lives in `src.residuals.series.compute_and_persist_series`, and is driven end-to-end by `run_me.py` (`_persist_spread_series` for a single sector, `_persist_series_multi` for many) as part of its `residuals` stage, right after the panel is built.